# Digikala Recommendation Status — Transformer Encoders

This notebook is the second experiment stage: a controlled comparison of **ParsBERT** and **XLM-RoBERTa-base** on the same sample and split as the classical baseline. Selection uses validation `Macro-F1`; test is not used until after model selection.

## Kaggle prerequisites

1. Select a GPU under `Settings > Accelerator`. T4/L4/A100 run directly; for P100, the first cell installs a compatible `cu126` build.
2. Enable Internet.
3. Prefer attaching the previous notebook output with `Add Input` so `sampled_split_manifest.csv` is found and the exact same records and split are used.
4. If the manifest is absent, the notebook reconstructs the sample and split from the pinned dataset with the same seed and logic.
5. If this notebook has run in the current session, select `Restart Session`, then run all cells from the beginning in order.

This version targets CUDA; TPU v5e requires a separate `torch_xla` rewrite. Defaults are conservative for 16 GB memory: batch size 8 and gradient accumulation 4. If both candidates take too long, set `CANDIDATE_EPOCHS` to 1 only for a smoke test; retain the default value 2 for reportable results.

In [1]:
from __future__ import annotations

import subprocess
import sys

# Run this cell in a fresh session before importing torch or transformers.
def subprocess_text(command):
    return subprocess.check_output(command, stderr=subprocess.DEVNULL, text=True).strip()

try:
    gpu_line = subprocess_text([
        'nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader,nounits'
    ]).splitlines()[0]
    gpu_name_before_import, capability_text = [part.strip() for part in gpu_line.rsplit(',', 1)]
    gpu_compute_capability = float(capability_text)
except Exception:
    gpu_name_before_import = 'unknown'
    gpu_compute_capability = None

try:
    installed_torch_cuda = subprocess_text([
        sys.executable, '-c', 'import torch; print(torch.version.cuda or \"cpu\")'
    ])
except Exception:
    installed_torch_cuda = 'missing'

legacy_gpu = (
    gpu_compute_capability is not None and gpu_compute_capability < 7.5
) or ('P100' in gpu_name_before_import or 'V100' in gpu_name_before_import)
needs_cu126 = legacy_gpu and not installed_torch_cuda.startswith('12.6')
torch_was_already_imported = 'torch' in sys.modules

print({
    'detected_gpu': gpu_name_before_import,
    'compute_capability': gpu_compute_capability,
    'installed_torch_cuda': installed_torch_cuda,
    'needs_cu126': needs_cu126,
})

if needs_cu126:
    print('Installing the PyTorch 2.10 CUDA 12.6 build required by Pascal/Volta GPUs ...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
        'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    if torch_was_already_imported:
        raise RuntimeError(
            'PyTorch cu126 was installed, but the previous build is still loaded. ' 
            'Restart the session now and rerun the notebook from the first cell.'
        )

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<5',
    'accelerate>=1.2,<2',
    'sentencepiece>=0.2',
    'safetensors>=0.4',
])

torch_build = subprocess_text([
    sys.executable, '-c',
    'import torch; print(torch.__version__, torch.version.cuda, \",\".join(torch.cuda.get_arch_list()), sep=\"|\")',
])
print('Runtime build:', torch_build)
print('Transformer dependencies are ready.')

{'detected_gpu': 'Tesla T4', 'compute_capability': 7.5, 'installed_torch_cuda': '12.8', 'needs_cu126': False}
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.6 MB/s eta 0:00:00
Runtime build: 2.10.0+cu128|12.8|sm_70,sm_75,sm_80,sm_86,sm_90,sm_100,sm_120
Transformer dependencies are ready.


In [2]:
import gc
import hashlib
import inspect
import json
import math
import os
import platform
import random
import socket
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
VALID_LABELS = ['recommended', 'not_recommended', 'no_idea']
LABEL2ID = {label: index for index, label in enumerate(VALID_LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}

HF_REPO_ID = 'RadeAI/Digikala_comments_products'
HF_REVISION = '89c3133b169c8d3793db8834f56f32fee33d9db0'
HF_FILENAME = 'digikala-comments.csv'
HF_EXPECTED_SIZE = 1_278_526_959
HF_EXPECTED_SHA256 = 'c7a8aa3020334fde8ec24944576a03fe5785e6fe12cd01042f5836632ddf8297'
HF_DOWNLOAD_URL = f'https://huggingface.co/datasets/{HF_REPO_ID}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true'

COMMENTS_PATH = os.getenv('DIGIKALA_COMMENTS_PATH') or None
MANIFEST_PATH = os.getenv('DIGIKALA_MANIFEST_PATH') or None
SAMPLE_FRACTION = 0.02
MAX_SAMPLED_ROWS = 150_000
CHUNK_SIZE = 250_000

MODEL_SPECS = [
    {'name': 'parsbert', 'checkpoint': 'HooshvareLab/bert-fa-base-uncased'},
    {'name': 'xlm_roberta_base', 'checkpoint': 'FacebookAI/xlm-roberta-base'},
]
CANDIDATE_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_LENGTH_CAP = 160
TOKEN_LENGTH_PROBE_ROWS = 20_000

BASELINE_VALIDATION_MACRO_F1 = 0.6806198033446752
BASELINE_TEST_MACRO_F1_REFERENCE = 0.6611412090831573
MIN_ABSOLUTE_VALIDATION_GAIN = 0.02
EXPECTED_SPLIT_PROFILE = {
    'train': {'rows': 85_694, 'text_groups': 66_899, 'recommended': 68_106, 'not_recommended': 8_159, 'no_idea': 9_429},
    'validation': {'rows': 9_941, 'text_groups': 8_249, 'recommended': 7_678, 'not_recommended': 993, 'no_idea': 1_270},
    'test': {'rows': 9_662, 'text_groups': 8_215, 'recommended': 7_612, 'not_recommended': 980, 'no_idea': 1_070},
}

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'outputs' / 'kaggle_transformers'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_DIR / 'transformer_candidate_runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('No GPU was found. Select a GPU under Kaggle Settings > Accelerator and restart the session.')

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = tuple(torch.cuda.get_device_capability(0))
compiled_cuda_arches = torch.cuda.get_arch_list()
device_arch = f'sm_{gpu_capability[0]}{gpu_capability[1]}'
if device_arch not in compiled_cuda_arches:
    raise RuntimeError(
        f'GPU {gpu_name} requires {device_arch}, but this PyTorch build supports {compiled_cuda_arches}. '
        'For P100/V100 restart the session and run the compatibility cell first.'
    )

# Execute a real kernel so CUDA incompatibility fails here with a clear message.
cuda_probe = torch.ones(8, device='cuda').sum()
torch.cuda.synchronize()
del cuda_probe

use_bf16 = gpu_capability[0] >= 8 and bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
use_fp16 = not use_bf16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print(json.dumps({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'scikit_learn': sklearn.__version__,
    'gpu': gpu_name,
    'gpu_compute_capability': gpu_capability,
    'torch_cuda_runtime': torch.version.cuda,
    'compiled_cuda_arches': compiled_cuda_arches,
    'precision': 'bf16' if use_bf16 else 'fp16',
    'output_dir': str(OUTPUT_DIR),
}, ensure_ascii=False, indent=2))

{
  "python": "3.12.13",
  "torch": "2.10.0+cu128",
  "transformers": "4.57.6",
  "scikit_learn": "1.6.1",
  "gpu": "Tesla T4",
  "gpu_compute_capability": [
    7,
    5
  ],
  "torch_cuda_runtime": "12.8",
  "compiled_cuda_arches": [
    "sm_70",
    "sm_75",
    "sm_80",
    "sm_86",
    "sm_90",
    "sm_100",
    "sm_120"
  ],
  "precision": "fp16",
  "output_dir": "/kaggle/working"
}


## Retrieve and validate the data source

The CSV is retrieved from an exact Hugging Face revision and verified by size and SHA-256, so later repository changes cannot affect this experiment.

In [3]:
def file_sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def validate_hf_file(path: Path, verify_hash: bool = True) -> Path:
    actual_size = path.stat().st_size
    if actual_size != HF_EXPECTED_SIZE:
        raise ValueError(f'Unexpected file size: {actual_size:,}; expected {HF_EXPECTED_SIZE:,}')
    if verify_hash:
        actual_hash = file_sha256(path)
        if actual_hash != HF_EXPECTED_SHA256:
            raise ValueError(f'SHA256 mismatch: {actual_hash}')
    return path

def assert_huggingface_network() -> None:
    try:
        socket.getaddrinfo('huggingface.co', 443, type=socket.SOCK_STREAM)
    except socket.gaierror as error:
        raise RuntimeError(
            'Kaggle Internet is unavailable. Enable Internet, restart the session, and rerun the notebook from the beginning.'
        ) from error

def direct_download_from_hf(target: Path) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + '.part')
    request = urllib.request.Request(HF_DOWNLOAD_URL, headers={'User-Agent': 'kaggle-digikala-transformer/1.0'})
    print('Direct download:', HF_DOWNLOAD_URL)
    downloaded = 0
    report_step = 128 * 1024 * 1024
    next_report = report_step
    with urllib.request.urlopen(request, timeout=120) as response, partial.open('wb') as output:
        while block := response.read(8 * 1024 * 1024):
            output.write(block)
            downloaded += len(block)
            if downloaded >= next_report:
                print(f'Downloaded: {downloaded / 1_000_000_000:.2f} GB')
                next_report += report_step
    partial.replace(target)
    return target

def get_comments_csv(manual_path: str | None = None) -> Path:
    if manual_path:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f'COMMENTS_PATH does not exist: {path}')
        print('Using manual/local CSV override.')
        return validate_hf_file(path)

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        input_candidates = [
            path for path in kaggle_input.rglob(HF_FILENAME)
            if path.is_file() and path.stat().st_size == HF_EXPECTED_SIZE
        ]
        for candidate in input_candidates:
            try:
                print('Validating dataset already attached as Kaggle Input:', candidate)
                return validate_hf_file(candidate)
            except ValueError:
                pass

    direct_target = OUTPUT_DIR / 'hf_data' / HF_FILENAME
    if direct_target.exists():
        try:
            return validate_hf_file(direct_target)
        except ValueError as error:
            print('Cached direct file is invalid; downloading again:', error)

    assert_huggingface_network()
    try:
        from huggingface_hub import hf_hub_download
        cached_path = Path(hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type='dataset',
            filename=HF_FILENAME,
            revision=HF_REVISION,
            cache_dir=str(OUTPUT_DIR / 'hf_cache'),
        ))
        return validate_hf_file(cached_path)
    except Exception as error:
        print(f'huggingface_hub failed ({type(error).__name__}: {error}); trying direct URL.')
        assert_huggingface_network()
        return validate_hf_file(direct_download_from_hf(direct_target))

comments_path = get_comments_csv(COMMENTS_PATH)
print('Comments CSV:', comments_path)
print('Verified size (GB):', round(comments_path.stat().st_size / 1_000_000_000, 3))
print('Verified SHA256:', HF_EXPECTED_SHA256)

digikala-comments.csv:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

Comments CSV: /kaggle/working/hf_cache/datasets--RadeAI--Digikala_comments_products/snapshots/89c3133b169c8d3793db8834f56f32fee33d9db0/digikala-comments.csv
Verified size (GB): 1.279
Verified SHA256: c7a8aa3020334fde8ec24944576a03fe5785e6fe12cd01042f5836632ddf8297


## Recover the same sample and split

The baseline manifest is searched first. If found, only its identifiers are read from the CSV. The fallback reproduces the first notebook's sampling, text cleaning, group key, and `StratifiedGroupKFold` logic. No target feature enters the text.

In [4]:
TEXT_COLUMNS = ['id', 'title', 'body', 'advantages', 'disadvantages', 'recommendation_status', 'product_id']
NULL_TOKENS = {'', 'nan', 'none', 'null', 'na', 'n/a'}
ARABIC_TO_PERSIAN = str.maketrans({'\u064a': '\u06cc', '\u0649': '\u06cc', '\u0643': '\u06a9'})

def normalize_text_series(series: pd.Series) -> pd.Series:
    out = series.fillna('').astype(str)
    stripped_lower = out.str.strip().str.lower()
    out = out.mask(stripped_lower.isin(NULL_TOKENS), '')
    out = out.str.normalize('NFKC').str.translate(ARABIC_TO_PERSIAN)
    out = out.str.replace('\ufeff', '', regex=False)
    out = out.str.replace(r'\s+', ' ', regex=True).str.strip()
    return out

def stable_sample_mask(ids: pd.Series, fraction: float, seed: int = 42) -> np.ndarray:
    sample_keys = ids.astype(str) + f'-{seed}'
    hashed = pd.util.hash_pandas_object(sample_keys, index=False).to_numpy(dtype=np.uint64)
    scale = np.uint64(1_000_000)
    threshold = int(round(fraction * int(scale)))
    return (hashed % scale) < threshold

def build_model_text(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in ['title', 'body', 'advantages', 'disadvantages']:
        frame[column] = normalize_text_series(frame[column])
    tagged_parts = []
    for column, tag in [('title', '[TITLE]'), ('body', '[BODY]'), ('advantages', '[ADVANTAGES]'), ('disadvantages', '[DISADVANTAGES]')]:
        tagged_parts.append(np.where(frame[column].ne(''), tag + ' ' + frame[column] + ' ', ''))
    full_text = pd.Series(tagged_parts[0], index=frame.index)
    for part in tagged_parts[1:]:
        full_text = full_text + pd.Series(part, index=frame.index)
    frame['text_full'] = full_text.str.replace(r'\s+', ' ', regex=True).str.strip()
    frame['text_body'] = frame['body']
    split_text = frame['text_body'].where(frame['text_body'].ne(''), frame['text_full'])
    frame['text_group_id'] = split_text.map(lambda value: hashlib.sha1(value.encode('utf-8')).hexdigest())
    return frame

def cap_sample_by_complete_groups(frame: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if max_rows <= 0 or len(frame) <= max_rows:
        return frame
    group_sizes = frame.groupby('text_group_id', sort=False).size().rename('rows').reset_index()
    order_hash = pd.util.hash_pandas_object(group_sizes['text_group_id'] + f'-{seed}', index=False).to_numpy(dtype=np.uint64)
    group_sizes = group_sizes.assign(order_hash=order_hash).sort_values('order_hash')
    selected = group_sizes.loc[group_sizes['rows'].cumsum() <= max_rows, 'text_group_id']
    if selected.empty:
        selected = group_sizes.head(1)['text_group_id']
    return frame[frame['text_group_id'].isin(set(selected))].copy()

def create_splits(frame: pd.DataFrame, seed: int = 42) -> dict[str, pd.DataFrame]:
    y = frame['recommendation_status'].to_numpy()
    groups = frame['text_group_id'].to_numpy()
    x = np.zeros(len(frame), dtype=np.uint8)
    outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
    train_val_idx, test_idx = next(outer.split(x, y, groups))
    train_val = frame.iloc[train_val_idx].copy()
    inner = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=seed + 1)
    train_rel_idx, val_rel_idx = next(inner.split(
        np.zeros(len(train_val), dtype=np.uint8),
        train_val['recommendation_status'].to_numpy(),
        train_val['text_group_id'].to_numpy(),
    ))
    return {
        'train': train_val.iloc[train_rel_idx].copy(),
        'validation': train_val.iloc[val_rel_idx].copy(),
        'test': frame.iloc[test_idx].copy(),
    }

def find_manifest(manual_path: str | None = None) -> Path | None:
    if manual_path:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f'MANIFEST_PATH does not exist: {path}')
        return path
    candidates = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]:
        if root.exists():
            candidates.extend(root.rglob('sampled_split_manifest.csv'))
    candidates = [path for path in candidates if path.resolve() != (OUTPUT_DIR / 'sampled_split_manifest.csv').resolve()]
    return sorted(candidates, key=lambda path: len(str(path)))[0] if candidates else None

def rows_for_manifest(csv_path: Path, manifest: pd.DataFrame) -> pd.DataFrame:
    wanted = set(manifest['id'].astype(str))
    chunks = []
    for chunk_number, chunk in enumerate(pd.read_csv(
        csv_path, usecols=TEXT_COLUMNS, dtype=str, chunksize=CHUNK_SIZE,
        keep_default_na=False, na_filter=False, encoding='utf-8-sig',
    ), start=1):
        chosen = chunk[chunk['id'].astype(str).isin(wanted)].copy()
        if not chosen.empty:
            chunks.append(chosen)
        if chunk_number % 5 == 0:
            print(f'Chunks: {chunk_number:,} | recovered rows: {sum(map(len, chunks)):,}/{len(wanted):,}')
    frame = pd.concat(chunks, ignore_index=True)
    frame = frame.drop_duplicates('id', keep='first')
    frame = build_model_text(frame)
    merged = manifest.merge(
        frame.drop(columns=['product_id', 'recommendation_status', 'text_group_id'], errors='ignore'),
        on='id', how='left', validate='one_to_one',
    )
    missing = int(merged['text_full'].isna().sum())
    if missing:
        raise RuntimeError(f'{missing} manifest ids were not recovered from the verified source CSV.')
    return merged

def rebuild_sample(csv_path: Path) -> tuple[pd.DataFrame, dict]:
    sampled_chunks = []
    scan_rows = 0
    valid_label_rows = 0
    for chunk_number, chunk in enumerate(pd.read_csv(
        csv_path, usecols=TEXT_COLUMNS, dtype=str, chunksize=CHUNK_SIZE,
        keep_default_na=False, na_filter=False, encoding='utf-8-sig',
    ), start=1):
        scan_rows += len(chunk)
        chunk['recommendation_status'] = chunk['recommendation_status'].astype(str).str.strip()
        chunk = chunk[chunk['recommendation_status'].isin(VALID_LABELS)].copy()
        valid_label_rows += len(chunk)
        if not chunk.empty:
            chunk = chunk.loc[stable_sample_mask(chunk['id'], SAMPLE_FRACTION, SEED)].copy()
            if not chunk.empty:
                sampled_chunks.append(chunk)
        if chunk_number % 5 == 0:
            print(f'Chunks: {chunk_number:,} | scanned: {scan_rows:,} | sampled: {sum(map(len, sampled_chunks)):,}')
    frame = pd.concat(sampled_chunks, ignore_index=True)
    duplicate_ids = int(frame.duplicated('id', keep='first').sum())
    frame = build_model_text(frame).drop_duplicates('id', keep='first')
    empty_texts = int(frame['text_full'].eq('').sum())
    frame = frame[frame['text_full'].ne('')].copy()
    frame = cap_sample_by_complete_groups(frame, MAX_SAMPLED_ROWS, SEED).reset_index(drop=True)
    return frame, {
        'physical_rows_scanned': int(scan_rows),
        'valid_label_rows': int(valid_label_rows),
        'duplicate_comment_id_rows_removed': duplicate_ids,
        'empty_text_rows_removed': empty_texts,
    }

In [5]:
manifest_source = find_manifest(MANIFEST_PATH)
rebuild_audit = {}

if manifest_source is not None:
    print('Using previous baseline manifest:', manifest_source)
    manifest = pd.read_csv(manifest_source, dtype=str, keep_default_na=False)
    required = {'id', 'product_id', 'text_group_id', 'recommendation_status', 'split'}
    if not required.issubset(manifest.columns):
        raise ValueError(f'Manifest is missing columns: {sorted(required - set(manifest.columns))}')
    if manifest['id'].duplicated().any():
        raise ValueError('Manifest contains duplicate comment ids.')
    if not set(manifest['split']).issubset({'train', 'validation', 'test'}):
        raise ValueError('Manifest contains an unknown split name.')
    if not set(manifest['recommendation_status']).issubset(VALID_LABELS):
        raise ValueError('Manifest contains an unknown target label.')
    sample = rows_for_manifest(comments_path, manifest)
    split_frames = {name: sample[sample['split'].eq(name)].copy() for name in ['train', 'validation', 'test']}
    split_source = 'previous_baseline_manifest'
else:
    print('Manifest not found; deterministically rebuilding the sample and split.')
    sample, rebuild_audit = rebuild_sample(comments_path)
    split_frames = create_splits(sample, SEED)
    split_source = 'deterministic_rebuild'

train_df = split_frames['train'].reset_index(drop=True)
val_df = split_frames['validation'].reset_index(drop=True)
test_df = split_frames['test'].reset_index(drop=True)

for frame in [train_df, val_df, test_df]:
    frame['label_id'] = frame['recommendation_status'].map(LABEL2ID).astype(int)

for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert set(split_frames[left]['id']).isdisjoint(split_frames[right]['id'])
    assert set(split_frames[left]['text_group_id']).isdisjoint(split_frames[right]['text_group_id'])

split_rows = []
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    counts = frame['recommendation_status'].value_counts()
    split_rows.append({
        'split': name,
        'rows': int(len(frame)),
        'text_groups': int(frame['text_group_id'].nunique()),
        **{label: int(counts.get(label, 0)) for label in VALID_LABELS},
    })
split_profile = pd.DataFrame(split_rows)
for row in split_rows:
    expected = EXPECTED_SPLIT_PROFILE[row['split']]
    observed = {key: row[key] for key in expected}
    if observed != expected:
        raise RuntimeError(
            f"Split {row['split']} does not match the baseline run. "
            f'Observed={observed}, expected={expected}. Attach the original sampled_split_manifest.csv.'
        )
display(split_profile)
print('Split source:', split_source)
print('Exact baseline profile and leakage checks passed.')

manifest_parts = []
for split_name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    part = frame[['id', 'product_id', 'text_group_id', 'recommendation_status']].copy()
    part['split'] = split_name
    manifest_parts.append(part)
manifest_output_path = OUTPUT_DIR / 'sampled_split_manifest.csv'
pd.concat(manifest_parts, ignore_index=True).to_csv(manifest_output_path, index=False)
print('Saved active manifest:', manifest_output_path)

Using previous baseline manifest: /kaggle/input/notebooks/maslri/digikala-classical-baselines/sampled_split_manifest.csv
Chunks: 5 | recovered rows: 21,626/105,297
Chunks: 10 | recovered rows: 42,931/105,297
Chunks: 15 | recovered rows: 64,119/105,297
Chunks: 20 | recovered rows: 85,553/105,297
Chunks: 25 | recovered rows: 105,355/105,297


,split,rows,text_groups,recommended,not_recommended,no_idea
0,train,85694,66899,68106,8159,9429
1,validation,9941,8249,7678,993,1270
2,test,9662,8215,7612,980,1070


Split source: previous_baseline_manifest
Exact baseline profile and leakage checks passed.
Saved active manifest: /kaggle/working/sampled_split_manifest.csv


## Dataset, metrics, and weighted Trainer

Class imbalance is handled with cross-entropy weights derived from train frequencies. Input length is probed separately for each tokenizer at the 99th percentile and capped at 160 tokens. Padding is dynamic within each batch.

In [6]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(map(int, labels))
        self.tokenizer = tokenizer
        self.max_length = int(max_length)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        item['labels'] = self.labels[index]
        return item

def calculate_metrics(y_true_ids, y_pred_ids) -> dict:
    y_true_ids = np.asarray(y_true_ids)
    y_pred_ids = np.asarray(y_pred_ids)
    result = {
        'macro_f1': float(f1_score(y_true_ids, y_pred_ids, labels=list(ID2LABEL), average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true_ids, y_pred_ids, labels=list(ID2LABEL), average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true_ids, y_pred_ids)),
    }
    per_class = classification_report(
        y_true_ids, y_pred_ids, labels=list(ID2LABEL), output_dict=True, zero_division=0
    )
    for label_id, label_name in ID2LABEL.items():
        row = per_class[str(label_id)]
        result[f'precision_{label_name}'] = float(row['precision'])
        result[f'recall_{label_name}'] = float(row['recall'])
        result[f'f1_{label_name}'] = float(row['f1-score'])
    return result

def trainer_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return calculate_metrics(labels, predictions)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device)
        loss = nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def class_weights_for(frame: pd.DataFrame) -> list[float]:
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(len(VALID_LABELS)),
        y=frame['label_id'].to_numpy(),
    )
    return [float(value) for value in weights]

def estimate_max_length(tokenizer, texts: pd.Series) -> tuple[int, dict]:
    probe = texts.sample(min(TOKEN_LENGTH_PROBE_ROWS, len(texts)), random_state=SEED).tolist()
    encoded = tokenizer(probe, truncation=False, padding=False, return_length=True)
    lengths = np.asarray(encoded['length'], dtype=np.int32)
    stats = {
        'probe_rows': int(len(lengths)),
        'p50': float(np.quantile(lengths, 0.50)),
        'p90': float(np.quantile(lengths, 0.90)),
        'p95': float(np.quantile(lengths, 0.95)),
        'p99': float(np.quantile(lengths, 0.99)),
        'max': int(lengths.max()),
    }
    chosen = max(64, min(MAX_LENGTH_CAP, int(math.ceil(stats['p99'] / 8) * 8)))
    return chosen, stats

def make_training_arguments(output_dir: Path, epochs: int, do_eval: bool) -> TrainingArguments:
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=int(epochs),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type='linear',
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=100,
        report_to='none',
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=2,
        save_total_limit=1,
        remove_unused_columns=True,
    )
    signature = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = 'eval_strategy' if 'eval_strategy' in signature else 'evaluation_strategy'
    if do_eval:
        kwargs.update({
            eval_key: 'epoch',
            'save_strategy': 'epoch',
            'load_best_model_at_end': True,
            'metric_for_best_model': 'macro_f1',
            'greater_is_better': True,
        })
    else:
        kwargs.update({eval_key: 'no', 'save_strategy': 'no'})
    return TrainingArguments(**kwargs)

def make_trainer(model, tokenizer, train_dataset, eval_dataset, output_dir, epochs, class_weights):
    arguments = make_training_arguments(output_dir, epochs, eval_dataset is not None)
    kwargs = dict(
        model=model,
        args=arguments,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer,
            pad_to_multiple_of=8,
            return_tensors='pt',
        ),
        compute_metrics=trainer_metrics if eval_dataset is not None else None,
        class_weights=class_weights,
    )
    trainer_signature = inspect.signature(Trainer.__init__).parameters
    if 'processing_class' in trainer_signature:
        kwargs['processing_class'] = tokenizer
    else:
        kwargs['tokenizer'] = tokenizer
    return WeightedTrainer(**kwargs)

## Train and compare candidates on validation

Each checkpoint is fine-tuned independently. Selection uses only `validation Macro-F1`, while per-class F1, especially `no_idea`, is also recorded. This cell may take several hours.

In [7]:
candidate_results = []
train_weights = class_weights_for(train_df)
print('Balanced class weights:', dict(zip(VALID_LABELS, train_weights)))

for spec in MODEL_SPECS:
    print('\n' + '=' * 90)
    print('Training candidate:', spec['name'], '|', spec['checkpoint'])
    print('=' * 90)
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(spec['checkpoint'], use_fast=True)
    max_length, length_stats = estimate_max_length(tokenizer, train_df['text_full'])
    print('Token length statistics:', length_stats)
    print('Selected max_length:', max_length)

    train_dataset = TextClassificationDataset(
        train_df['text_full'], train_df['label_id'], tokenizer, max_length
    )
    validation_dataset = TextClassificationDataset(
        val_df['text_full'], val_df['label_id'], tokenizer, max_length
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        spec['checkpoint'],
        num_labels=len(VALID_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    resolved_revision = getattr(model.config, '_commit_hash', None)
    parameter_count = int(sum(parameter.numel() for parameter in model.parameters()))
    run_dir = RUNS_DIR / spec['name']
    trainer = make_trainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_dataset, eval_dataset=validation_dataset,
        output_dir=run_dir, epochs=CANDIDATE_EPOCHS, class_weights=train_weights,
    )

    fit_start = time.perf_counter()
    trainer.train()
    fit_seconds = time.perf_counter() - fit_start
    predict_start = time.perf_counter()
    validation_output = trainer.predict(validation_dataset)
    predict_seconds = time.perf_counter() - predict_start
    validation_prediction = np.argmax(validation_output.predictions, axis=-1)
    metrics = calculate_metrics(val_df['label_id'], validation_prediction)

    eval_logs = [row for row in trainer.state.log_history if 'eval_macro_f1' in row]
    best_log = max(eval_logs, key=lambda row: row['eval_macro_f1']) if eval_logs else {'epoch': CANDIDATE_EPOCHS}
    best_epoch = max(1, int(round(float(best_log.get('epoch', CANDIDATE_EPOCHS)))))
    result = {
        'model': spec['name'],
        'checkpoint': spec['checkpoint'],
        'resolved_revision': resolved_revision,
        'parameter_count': parameter_count,
        'max_length': int(max_length),
        'best_epoch': best_epoch,
        'fit_seconds': float(fit_seconds),
        'validation_predict_seconds': float(predict_seconds),
        **metrics,
        'token_length_stats': length_stats,
    }
    candidate_results.append(result)
    display(pd.DataFrame([{key: value for key, value in result.items() if key != 'token_length_stats'}]))

    del trainer, model, tokenizer, train_dataset, validation_dataset, validation_output, validation_prediction
    gc.collect()
    torch.cuda.empty_cache()

validation_leaderboard = pd.DataFrame([
    {key: value for key, value in row.items() if key != 'token_length_stats'}
    for row in candidate_results
]).sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(validation_leaderboard)

Balanced class weights: {'recommended': 0.41941483373956284, 'not_recommended': 3.501000939657638, 'no_idea': 3.0294481563969313}

Training candidate: parsbert | HooshvareLab/bert-fa-base-uncased


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Token length statistics: {'probe_rows': 20000, 'p50': 16.0, 'p90': 45.0, 'p95': 60.0, 'p99': 100.0, 'max': 646}
Selected max_length: 104


pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1,Accuracy,Precision Recommended,Recall Recommended,F1 Recommended,Precision Not Recommended,Recall Not Recommended,F1 Not Recommended,Precision No Idea,Recall No Idea,F1 No Idea
1,0.562200,0.563672,0.697326,0.817728,0.793582,0.973639,0.812972,0.886081,0.680637,0.817724,0.742909,0.357296,0.657480,0.462989
2,0.507800,0.558051,0.733375,0.849030,0.835027,0.963492,0.869628,0.914157,0.741722,0.789527,0.764878,0.429887,0.661417,0.521092


,model,checkpoint,resolved_revision,parameter_count,max_length,best_epoch,fit_seconds,validation_predict_seconds,macro_f1,weighted_f1,accuracy,precision_recommended,recall_recommended,f1_recommended,precision_not_recommended,recall_not_recommended,f1_not_recommended,precision_no_idea,recall_no_idea,f1_no_idea
0,xlm_roberta_base,FacebookAI/xlm-roberta-base,e73636d4f797dec63c3081bb6ed5c7b0bb3f2089,278045955,128,2,3628.901725,52.537494,0.733375,0.84903,0.835027,0.963492,0.869628,0.914157,0.741722,0.789527,0.764878,0.429887,0.661417,0.521092


,model,checkpoint,resolved_revision,parameter_count,max_length,best_epoch,fit_seconds,validation_predict_seconds,macro_f1,weighted_f1,accuracy,precision_recommended,recall_recommended,f1_recommended,precision_not_recommended,recall_not_recommended,f1_not_recommended,precision_no_idea,recall_no_idea,f1_no_idea
0,xlm_roberta_base,FacebookAI/xlm-roberta-base,e73636d4f797dec63c3081bb6ed5c7b0bb3f2089,278045955,128,2,3628.901725,52.537494,0.733375,0.849030,0.835027,0.963492,0.869628,0.914157,0.741722,0.789527,0.764878,0.429887,0.661417,0.521092
1,parsbert,HooshvareLab/bert-fa-base-uncased,a04aa40c97bcdde570ae11986a534542c2995a62,162843651,104,2,2428.826407,37.144251,0.721923,0.842941,0.827281,0.963658,0.863376,0.910765,0.731214,0.764350,0.747415,0.413043,0.658268,0.507590


## Promotion gate and final evaluation

A model reaches test only when its validation `Macro-F1` exceeds the baseline by at least 0.02. Otherwise the result is saved without touching test and the classical baseline remains preferred. If the gate passes, the winning checkpoint is retrained on train plus validation for the selected epoch count and evaluated on test exactly once.

In [8]:
best_candidate = max(candidate_results, key=lambda row: row['macro_f1'])
validation_gain = float(best_candidate['macro_f1'] - BASELINE_VALIDATION_MACRO_F1)
promoted = bool(validation_gain >= MIN_ABSOLUTE_VALIDATION_GAIN)

print('Selected candidate:', best_candidate['model'])
print(f"Validation Macro-F1: {best_candidate['macro_f1']:.4f}")
print(f'Baseline Validation Macro-F1: {BASELINE_VALIDATION_MACRO_F1:.4f}')
print(f'Absolute gain: {validation_gain:+.4f}')
print('Promotion threshold:', MIN_ABSOLUTE_VALIDATION_GAIN)
print('Promoted to final test:', promoted)

test_metrics = None
test_per_class = None
test_confusion_matrix = None
final_fit_seconds = None
test_predict_seconds = None
final_model_dir = OUTPUT_DIR / 'best_transformer_encoder'

if promoted:
    set_seed(SEED)
    train_val_df = pd.concat([train_df, val_df], ignore_index=True)
    tokenizer = AutoTokenizer.from_pretrained(best_candidate['checkpoint'], use_fast=True)
    max_length = int(best_candidate['max_length'])
    train_val_dataset = TextClassificationDataset(
        train_val_df['text_full'], train_val_df['label_id'], tokenizer, max_length
    )
    test_dataset = TextClassificationDataset(
        test_df['text_full'], test_df['label_id'], tokenizer, max_length
    )
    final_model = AutoModelForSequenceClassification.from_pretrained(
        best_candidate['checkpoint'],
        num_labels=len(VALID_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    final_trainer = make_trainer(
        model=final_model, tokenizer=tokenizer,
        train_dataset=train_val_dataset, eval_dataset=None,
        output_dir=OUTPUT_DIR / 'final_training_run',
        epochs=int(best_candidate['best_epoch']),
        class_weights=class_weights_for(train_val_df),
    )
    fit_start = time.perf_counter()
    final_trainer.train()
    final_fit_seconds = float(time.perf_counter() - fit_start)

    predict_start = time.perf_counter()
    test_output = final_trainer.predict(test_dataset)
    test_predict_seconds = float(time.perf_counter() - predict_start)
    test_prediction = np.argmax(test_output.predictions, axis=-1)
    test_metrics = calculate_metrics(test_df['label_id'], test_prediction)
    report = classification_report(
        test_df['label_id'], test_prediction, labels=list(ID2LABEL), output_dict=True, zero_division=0
    )
    test_per_class = {ID2LABEL[index]: report[str(index)] for index in ID2LABEL}
    test_confusion_matrix = confusion_matrix(
        test_df['label_id'], test_prediction, labels=list(ID2LABEL)
    ).tolist()

    final_model_dir.mkdir(parents=True, exist_ok=True)
    final_trainer.save_model(str(final_model_dir))
    tokenizer.save_pretrained(str(final_model_dir))
    (final_model_dir / 'inference_config.json').write_text(json.dumps({
        'max_length': max_length,
        'labels': VALID_LABELS,
        'normalization_version': 'fa_light_v1',
        'text_column': 'text_full',
    }, ensure_ascii=False, indent=2), encoding='utf-8')

    print('FINAL TEST metrics:')
    print(json.dumps(test_metrics, ensure_ascii=False, indent=2))
    print('Per-class report:')
    print(json.dumps(test_per_class, ensure_ascii=False, indent=2))
else:
    print('The Transformer did not meet the improvement gate; test was not evaluated and the classical baseline remains preferred.')

Selected candidate: xlm_roberta_base
Validation Macro-F1: 0.7334
Baseline Validation Macro-F1: 0.6806
Absolute gain: +0.0528
Promotion threshold: 0.02
Promoted to final test: True


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,1.086500
200,0.899900
300,0.722100
400,0.643800
500,0.626600
600,0.589200
700,0.622600
800,0.612500
900,0.582400
1000,0.606900


FINAL TEST metrics:
{
  "macro_f1": 0.7174867593282346,
  "weighted_f1": 0.8554543471635552,
  "accuracy": 0.8423721796729455,
  "precision_recommended": 0.9604214123006833,
  "recall_recommended": 0.8862322648449816,
  "f1_recommended": 0.9218365673681334,
  "precision_not_recommended": 0.7438905180840665,
  "recall_not_recommended": 0.7765306122448979,
  "f1_not_recommended": 0.7598602096854717,
  "precision_no_idea": 0.3913312693498452,
  "recall_no_idea": 0.5906542056074766,
  "f1_no_idea": 0.4707635009310987
}
Per-class report:
{
  "recommended": {
    "precision": 0.9604214123006833,
    "recall": 0.8862322648449816,
    "f1-score": 0.9218365673681334,
    "support": 7612.0
  },
  "not_recommended": {
    "precision": 0.7438905180840665,
    "recall": 0.7765306122448979,
    "f1-score": 0.7598602096854717,
    "support": 980.0
  },
  "no_idea": {
    "precision": 0.3913312693498452,
    "recall": 0.5906542056074766,
    "f1-score": 0.4707635009310987,
    "support": 1070.0
  }
}


In [9]:
validation_results_path = OUTPUT_DIR / 'transformer_validation_results.csv'
summary_path = OUTPUT_DIR / 'transformer_run_summary.json'
validation_leaderboard.to_csv(validation_results_path, index=False)

summary = {
    'task': 'digikala_recommendation_status_transformer_encoders',
    'seed': SEED,
    'huggingface_repo': HF_REPO_ID,
    'huggingface_revision': HF_REVISION,
    'huggingface_filename': HF_FILENAME,
    'source_sha256': HF_EXPECTED_SHA256,
    'source_file': str(comments_path),
    'source_size_bytes': int(comments_path.stat().st_size),
    'split_source': split_source,
    'manifest_source': str(manifest_source) if manifest_source else None,
    'sample_fraction': SAMPLE_FRACTION,
    'max_sampled_rows': MAX_SAMPLED_ROWS,
    'split_profile': split_profile.to_dict(orient='records'),
    'rebuild_audit': rebuild_audit,
    'training_config': {
        'candidate_epochs': CANDIDATE_EPOCHS,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'effective_train_batch_size': TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'max_length_cap': MAX_LENGTH_CAP,
        'class_weighting': 'balanced_train_frequency',
    },
    'baseline_validation_macro_f1': BASELINE_VALIDATION_MACRO_F1,
    'baseline_test_macro_f1_reference_only': BASELINE_TEST_MACRO_F1_REFERENCE,
    'minimum_absolute_validation_gain': MIN_ABSOLUTE_VALIDATION_GAIN,
    'validation_candidates': candidate_results,
    'selected_model': best_candidate['model'],
    'selected_checkpoint': best_candidate['checkpoint'],
    'selected_resolved_revision': best_candidate['resolved_revision'],
    'selected_validation_macro_f1': best_candidate['macro_f1'],
    'absolute_validation_gain': validation_gain,
    'promoted_to_test': promoted,
    'final_fit_seconds': final_fit_seconds,
    'test_predict_seconds': test_predict_seconds,
    'test_metrics': test_metrics,
    'test_per_class': test_per_class,
    'test_confusion_matrix_label_order': VALID_LABELS if promoted else None,
    'test_confusion_matrix': test_confusion_matrix,
    'final_model_dir': str(final_model_dir) if promoted else None,
    'versions': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'platform': platform.platform(),
        'gpu': gpu_name,
        'gpu_compute_capability': list(gpu_capability),
        'torch_cuda_runtime': torch.version.cuda,
        'compiled_cuda_arches': compiled_cuda_arches,
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved artifacts:')
for path in [validation_results_path, manifest_output_path, summary_path]:
    print(f' - {path} ({path.stat().st_size / 1_000_000:.2f} MB)')
if promoted:
    model_size = sum(path.stat().st_size for path in final_model_dir.rglob('*') if path.is_file())
    print(f' - {final_model_dir} ({model_size / 1_000_000:.2f} MB)')

print('\n' + '#' * 28 + ' COPY THIS SUMMARY ' + '#' * 28)
print(json.dumps(summary, ensure_ascii=False, indent=2))

Saved artifacts:
 - /kaggle/working/transformer_validation_results.csv (0.00 MB)
 - /kaggle/working/sampled_split_manifest.csv (8.01 MB)
 - /kaggle/working/transformer_run_summary.json (0.01 MB)
 - /kaggle/working/best_transformer_encoder (1134.37 MB)

############################ COPY THIS SUMMARY ############################
{
  "task": "digikala_recommendation_status_transformer_encoders",
  "seed": 42,
  "huggingface_repo": "RadeAI/Digikala_comments_products",
  "huggingface_revision": "89c3133b169c8d3793db8834f56f32fee33d9db0",
  "huggingface_filename": "digikala-comments.csv",
  "source_sha256": "c7a8aa3020334fde8ec24944576a03fe5785e6fe12cd01042f5836632ddf8297",
  "source_file": "/kaggle/working/hf_cache/datasets--RadeAI--Digikala_comments_products/snapshots/89c3133b169c8d3793db8834f56f32fee33d9db0/digikala-comments.csv",
  "source_size_bytes": 1278526959,
  "split_source": "previous_baseline_manifest",
  "manifest_source": "/kaggle/input/notebooks/maslri/digikala-classical-basel

## Manual test of the selected model (only after promotion)

In [10]:
def prepare_one_text(title='', body='', advantages='', disadvantages=''):
    frame = pd.DataFrame([{
        'title': title, 'body': body,
        'advantages': advantages, 'disadvantages': disadvantages,
    }])
    return build_model_text(frame).iloc[0]['text_full']

manual_examples = [
    {'title': 'excellent', 'body': 'excellent quality; I would buy it again'},
    {'title': 'do not buy', 'body': 'the quality was poor and I returned it'},
    {'title': 'average', 'body': 'reasonable for the price, but I expected more'},
]

if promoted:
    manual_texts = [prepare_one_text(**example) for example in manual_examples]
    encoded = tokenizer(
        manual_texts, truncation=True, max_length=int(best_candidate['max_length']),
        padding=True, return_tensors='pt',
    ).to(final_model.device)
    final_model.eval()
    with torch.no_grad():
        logits = final_model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1).cpu().numpy()
    rows = []
    for example, probs in zip(manual_examples, probabilities):
        rows.append({
            **example,
            'prediction': ID2LABEL[int(probs.argmax())],
            **{f'p_{label}': float(probs[index]) for index, label in ID2LABEL.items()},
        })
    display(pd.DataFrame(rows))
else:
    print('No model was promoted to test and final saving.')

,title,body,prediction,p_recommended,p_not_recommended,p_no_idea
0,excellent,excellent quality; I would buy it again,recommended,0.990006,0.002800,0.007194
1,do not buy,the quality was poor and I returned it,not_recommended,0.001180,0.984620,0.014200
2,average,"reasonable for the price, but I expected more",no_idea,0.091919,0.029784,0.878297


## Outputs to retain

After the run, retain the complete `COPY THIS SUMMARY` block and the following Output artifacts:

- `transformer_run_summary.json`
- `transformer_validation_results.csv`
- `sampled_split_manifest.csv`
- the `best_transformer_encoder` directory only when `promoted_to_test=true`

If CUDA runs out of memory, restart the session, set `TRAIN_BATCH_SIZE=4` and `GRADIENT_ACCUMULATION_STEPS=8`, then rerun from the beginning; the effective batch size remains 32.